In [0]:
# 1) IMPORTS & LOGGING
import json
import logging
import requests
import yaml
from string import Template
from datetime import datetime, date
from typing import List, Dict, Tuple
from requests.adapters import HTTPAdapter, Retry
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, current_timestamp

In [0]:

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("compass.apple")

storage_account_name = "compassdataprod"
container = "sa-compasslake"
secret_scope_name = "storage_data"
secret_key_name   = "adlsstoragekeydata"

# Recupera o SAS Token do secret scope
sas_token = dbutils.secrets.get(scope=secret_scope_name, key=secret_key_name)

# Configuração com SAS Token
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account_name}.dfs.core.windows.net", sas_token)

# Teste de listagem
path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/raw_compass/apple_store/reviews/2025-09-02/"
# Lista arquivos e pastas em um diretório
files = dbutils.fs.ls(path)

# Exibe os arquivos
for f in files:
    print(f.name, f.path, f.size)


In [0]:
df = spark.read.table("metadata_compass.data_params")
display(df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType, DoubleType, MapType
from datetime import datetime

def create_or_append_contract(delta_table_path: str,
                              db_name: str,
                              table_name_control: str,
                              version: float,
                              source_layer: str,
                              table_name_target: str,
                              schema_array: list,
                              schema_target: list,
                              schema_depara: list,
                              regras_array: list,
                              source_config: dict,
                              target_config: dict,
                              fallback_config: dict):
    """
    Cria a tabela Delta caso não exista e insere um contrato de dados,
    incluindo configs de fonte, destino e fallback.
    Se a tabela já existir, apenas insere os dados.
    """

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

    # Verifica se a tabela existe
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {db_name}.{table_name_control}")
        table_exists = True
    except Exception:
        table_exists = False

    # Schema Delta nativo
    delta_schema = StructType([
        StructField("version", DoubleType(), True),
        StructField("source_layer", StringType(), True),
        StructField("table_name_target", StringType(), True),
        StructField("schema_expected", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True),
                StructField("other", StringType(), True) 
            ])
        ), True),
        StructField("schema_target", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True)
            ])
        ), True),
        StructField("schema_depara", ArrayType(
            StructType([
                StructField("source_column", StringType(), True),
                StructField("target_column", StringType(), True)
            ])
        ), True),
        StructField("rule_control", ArrayType(
            StructType([
                StructField("rule", StringType(), True),
                StructField("value", StringType(), True)
            ])
        ), True),
        StructField("source_config", MapType(StringType(), StringType()), True),
        StructField("target_config", MapType(StringType(), StringType()), True),
        StructField("fallback_config", MapType(StringType(), StringType()), True),
        StructField("last_modified", TimestampType(), True)
    ])

    if not table_exists:
        print(f"Tabela {db_name}.{table_name_control} não existe. Criando...")

        # DataFrame vazio para inicializar Delta log
        df_empty = spark.createDataFrame([], delta_schema)
        df_empty.write.format("delta").mode("overwrite").save(delta_table_path)

        # Cria tabela no metastore (sem comentários)
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {db_name}.{table_name_control}
        USING DELTA
        LOCATION '{delta_table_path}'
        """
        spark.sql(create_sql)
        print(f"Tabela {db_name}.{table_name_control} criada com sucesso.")

    else:
        print(f"Tabela {db_name}.{table_name_control} já existe. Apenas inserindo dados...")

    # Inserção dos dados
    data = [(version, source_layer, table_name_target, schema_array, schema_target, schema_depara, regras_array,
             source_config, target_config, fallback_config, datetime.now())]
    df_insert = spark.createDataFrame(data, delta_schema)
    df_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{db_name}.{table_name_control}")

    print(f"Contrato inserido com sucesso na tabela {db_name}.{table_name_control}.")

schema_array = [
    {"name_column": "submission_date", "type_column": "TIMESTAMP", "other": ""},
    {"name_column": "client.segment", "type_column": "STRING", "other": ""},
    {"name_column": "client.identification", "type_column": "STRING", "other": ""},
    {"name_column": "client.classification", "type_column": "STRING", "other": ""},
    {"name_column": "feedback_rating", "type_column": "INT", "other": ""},
    {"name_column": "feedback_comment", "type_column": "STRING", "other": ""},
    {"name_column": "service_type", "type_column": "STRING", "other": ""},
    {"name_column": "service_id", "type_column": "STRING", "other": ""},
    {"name_column": "feedback_specific_to_service", "type_column": "STRING", "other": ""},
    {"name_column": "source_channel", "type_column": "STRING", "other": ""},
    {"name_column": "source_id", "type_column": "STRING", "other": ""}, 
    {"name_column": "app_reference", "type_column": "STRING", "other": ""}, 
    {"name_column": "source_user_agent", "type_column": "STRING", "other": ""}
]

schema_target = [
    {"name_column": "submission_date", "type_column": "TIMESTAMP", "comment": "Data de submissão do feedback"},
    {"name_column": "client_id", "type_column": "STRING", "comment": "Identificação única do cliente"},
    {"name_column": "segment", "type_column": "STRING", "comment": "Segmento do cliente (pf/pj)"},
    {"name_column": "classification", "type_column": "STRING", "comment": "Classificação do cliente"},
    {"name_column": "feedback_rating", "type_column": "INT", "comment": "Nota atribuída"},
    {"name_column": "feedback_comment", "type_column": "STRING", "comment": "Comentário livre"},
    {"name_column": "service_type", "type_column": "STRING", "comment": "Tipo de serviço avaliado"},
    {"name_column": "service_id", "type_column": "STRING", "comment": "ID do serviço"},
    {"name_column": "feedback_specific_to_service", "type_column": "STRING", "comment": "Feedback específico do serviço"},
    {"name_column": "source_channel", "type_column": "STRING", "comment": "Canal de origem"},
    {"name_column": "app_reference", "type_column": "STRING", "comment": "App origem"},
    {"name_column": "source_id", "type_column": "STRING", "comment": "Identificador de origem"},
    {"name_column": "user_agent", "type_column": "STRING", "comment": "User agent da sessão"},
    {"name_column": "ingestion_ts", "type_column": "TIMESTAMP", "comment": "Timestamp de ingestão"},
    {"name_column": "date_load", "type_column": "STRING", "comment": "Data da carga do dado"}
]

schema_depara = [
    {"source_column": "submission_date", "target_column": "submission_date"},
    {"source_column": "client.identification", "target_column": "client_id"},
    {"source_column": "client.segment", "target_column": "segment"},
    {"source_column": "client.classification", "target_column": "classification"},
    {"source_column": "feedback_rating", "target_column": "feedback_rating"},
    {"source_column": "feedback_comment", "target_column": "feedback_comment"},
    {"source_column": "service_type", "target_column": "service_type"},
    {"source_column": "service_id", "target_column": "service_id"},
    {"source_column": "feedback_specific_to_service", "target_column": "feedback_specific_to_service"},
    {"source_column": "source_channel", "target_column": "source_channel"},
    {"source_column": "source_id", "target_column": "source_id"},
    {"source_column": "app_reference", "target_column": "app_reference"},
    {"source_column": "source_user_agent", "target_column": "user_agent"}
]

regras_array = [
    
        {"rule": "not_empty", "value": "true"},
        {"rule": "evolution_mergeschema", "value": "false"}
   
]

source_config = {
    "directory": "raw_compass/internal_db/reviews/",
    "format": "json"
}

target_config = {
    "path_blob": "abfss://b-compass@compassdataprod.dfs.core.windows.net/",
    "directory": "b-compass",
    "format": "delta",
    "mode": "append",
    "partitionBy": "date_load"
}

fallback_config = {
    "create_empty_if_missing": "true"
}


# Parametros indicador para a tabela de controle
container_system = "system-compass"
db_name = "metadata_compass"
table_name = "data_params"

delta_table_path = f"abfss://{container_system}@compassdataprod.dfs.core.windows.net/{container_system}/{table_name}/"

create_or_append_contract(
    delta_table_path=delta_table_path,
    db_name=db_name,
    table_name_control=table_name,
    version=1.1,
    source_layer="raw",
    table_name_target="internal_db",
    schema_array=schema_array,
    schema_target=schema_target,
    schema_depara=schema_depara,
    regras_array=regras_array,
    source_config=source_config,
    target_config=target_config,
    fallback_config=fallback_config
)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType, DoubleType, MapType
from datetime import datetime

def create_or_append_contract(delta_table_path: str,
                              db_name: str,
                              table_name_control: str,
                              version: float,
                              source_layer: str,
                              table_name_target: str,
                              schema_array: list,
                              schema_target: list,
                              schema_depara: list,
                              regras_array: list,
                              source_config: dict,
                              target_config: dict,
                              fallback_config: dict):
    """
    Cria a tabela Delta caso não exista e insere um contrato de dados,
    incluindo configs de fonte, destino e fallback.
    Se a tabela já existir, apenas insere os dados.
    """

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

    # Verifica se a tabela existe
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {db_name}.{table_name_control}")
        table_exists = True
    except Exception:
        table_exists = False

    # Schema Delta nativo
    delta_schema = StructType([
        StructField("version", DoubleType(), True),
        StructField("source_layer", StringType(), True),
        StructField("table_name_target", StringType(), True),
        StructField("schema_expected", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True),
                StructField("other", StringType(), True) 
            ])
        ), True),
        StructField("schema_target", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True)
            ])
        ), True),
        StructField("schema_depara", ArrayType(
            StructType([
                StructField("source_column", StringType(), True),
                StructField("target_column", StringType(), True)
            ])
        ), True),
        StructField("rule_control", ArrayType(
            StructType([
                StructField("rule", StringType(), True),
                StructField("value", StringType(), True)
            ])
        ), True),
        StructField("source_config", MapType(StringType(), StringType()), True),
        StructField("target_config", MapType(StringType(), StringType()), True),
        StructField("fallback_config", MapType(StringType(), StringType()), True),
        StructField("last_modified", TimestampType(), True)
    ])

    if not table_exists:
        print(f"Tabela {db_name}.{table_name_control} não existe. Criando...")

        # DataFrame vazio para inicializar Delta log
        df_empty = spark.createDataFrame([], delta_schema)
        df_empty.write.format("delta").mode("overwrite").save(delta_table_path)

        # Cria tabela no metastore (sem comentários)
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {db_name}.{table_name_control}
        USING DELTA
        LOCATION '{delta_table_path}'
        """
        spark.sql(create_sql)
        print(f"Tabela {db_name}.{table_name_control} criada com sucesso.")

    else:
        print(f"Tabela {db_name}.{table_name_control} já existe. Apenas inserindo dados...")

    # Inserção dos dados
    data = [(version, source_layer, table_name_target, schema_array, schema_target, schema_depara, regras_array,
             source_config, target_config, fallback_config, datetime.now())]
    df_insert = spark.createDataFrame(data, delta_schema)
    df_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{db_name}.{table_name_control}")

    print(f"Contrato inserido com sucesso na tabela {db_name}.{table_name_control}.")

schema_array = [
        {"name_column":"author_uri","type_column":"STRING","other":""},
        {"name_column":"author_name","type_column":"STRING","other":""},
        {"name_column":"author_label","type_column":"STRING","other":""},
        {"name_column":"updated_at","type_column":"STRING","other":""},
        {"name_column":"rating","type_column":"STRING","other":""},
        {"name_column":"version","type_column":"STRING","other":""},
        {"name_column":"review_id","type_column":"STRING","other":""},
        {"name_column":"title","type_column":"STRING","other":""},
        {"name_column":"content","type_column":"STRING","other":""},
        {"name_column":"content_type_attribute","type_column":"STRING","other":""},
        {"name_column":"link_rel","type_column":"STRING","other":""},
        {"name_column":"link_href","type_column":"STRING","other":""},
        {"name_column":"vote_sum","type_column":"STRING","other":""},
        {"name_column":"content_term","type_column":"STRING","other":""},
        {"name_column":"content_label","type_column":"STRING","other":""},
        {"name_column":"vote_count","type_column":"STRING","other":""}
    ]

schema_target = [
        {"name_column":"author_ura","type_column":"STRING"},
        {"name_column":"author_name","type_column":"STRING"},
        {"name_column":"author_label","type_column":"STRING"},
        {"name_column":"updated_at","type_column":"STRING"},
        {"name_column":"rating","type_column":"INT"},
        {"name_column":"version","type_column":"STRING"},
        {"name_column":"review_id","type_column":"STRING"},
        {"name_column":"title","type_column":"STRING"},
        {"name_column":"content","type_column":"STRING"},
        {"name_column":"content_type_attribute","type_column":"STRING"},
        {"name_column":"link_rel","type_column":"STRING"},
        {"name_column":"link_href","type_column":"STRING"},
        {"name_column":"vote_sum","type_column":"INT"},
        {"name_column":"content_term","type_column":"STRING"},
        {"name_column":"content_label","type_column":"STRING"},
        {"name_column":"vote_count","type_column":"INT"},{"name_column":"date_load","type_column":"STRING"},
        {"name_column":"app_reference","type_column":"STRING"},
        {"name_column":"ingestion_ts","type_column":"TIMESTAMP"},
        {"name_column":"date_load","type_column":"STRING"}
    ]

schema_depara = [
        {"source_column":"author_uri","target_column":"author_ura"},
        {"source_column":"author_name","target_column":"author_name"},
        {"source_column":"author_label","target_column":"author_label"},
        {"source_column":"updated_at","target_column":"updated_at"},
        {"source_column":"rating","target_column":"rating"},
        {"source_column":"version","target_column":"version"},
        {"source_column":"review_id","target_column":"review_id"},
        {"source_column":"title","target_column":"title"},
        {"source_column":"content","target_column":"content"},
        {"source_column":"content_type_attribute","target_column":"content_type_attribute"},
        {"source_column":"link_rel","target_column":"link_rel"},
        {"source_column":"link_href","target_column":"link_href"},
        {"source_column":"vote_sum","target_column":"vote_sum"},
        {"source_column":"content_term","target_column":"content_term"},
        {"source_column":"content_label","target_column":"content_label"},
        {"source_column":"vote_count","target_column":"vote_count"}]

regras_array = [
    
        {"rule": "not_empty", "value": "true"},
        {"rule": "evolution_mergeschema", "value": "false"}
   
]

source_config = {
    "directory": "raw_compass/internal_db/reviews/",
    "format": "json"
}

target_config = {
    "path_blob": "abfss://b-compass@compassdataprod.dfs.core.windows.net/",
    "directory": "b-compass",
    "format": "delta",
    "mode": "append",
    "partitionBy": "date_load"
}

fallback_config = {
    "create_empty_if_missing": "true"
}


# Parametros indicador para a tabela de controle
container_system = "system-compass"
db_name = "metadata_compass"
table_name = "data_params"

delta_table_path = f"abfss://{container_system}@compassdataprod.dfs.core.windows.net/{container_system}/{table_name}/"

create_or_append_contract(
    delta_table_path=delta_table_path,
    db_name=db_name,
    table_name_control=table_name,
    version=1.1,
    source_layer="raw",
    table_name_target="apple_reviews",
    schema_array=schema_array,
    schema_target=schema_target,
    schema_depara=schema_depara,
    regras_array=regras_array,
    source_config=source_config,
    target_config=target_config,
    fallback_config=fallback_config
)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType, DoubleType, MapType
from datetime import datetime

def create_or_append_contract(delta_table_path: str,
                              db_name: str,
                              table_name_control: str,
                              version: float,
                              source_layer: str,
                              table_name_target: str,
                              schema_array: list,
                              schema_target: list,
                              schema_depara: list,
                              regras_array: list,
                              source_config: dict,
                              target_config: dict,
                              fallback_config: dict):
    """
    Cria a tabela Delta caso não exista e insere um contrato de dados,
    incluindo configs de fonte, destino e fallback.
    Se a tabela já existir, apenas insere os dados.
    """

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

    # Verifica se a tabela existe
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {db_name}.{table_name_control}")
        table_exists = True
    except Exception:
        table_exists = False

    # Schema Delta nativo
    delta_schema = StructType([
        StructField("version", DoubleType(), True),
        StructField("source_layer", StringType(), True),
        StructField("table_name_target", StringType(), True),
        StructField("schema_expected", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True),
                StructField("other", StringType(), True) 
            ])
        ), True),
        StructField("schema_target", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True)
            ])
        ), True),
        StructField("schema_depara", ArrayType(
            StructType([
                StructField("source_column", StringType(), True),
                StructField("target_column", StringType(), True)
            ])
        ), True),
        StructField("rule_control", ArrayType(
            StructType([
                StructField("rule", StringType(), True),
                StructField("value", StringType(), True)
            ])
        ), True),
        StructField("source_config", MapType(StringType(), StringType()), True),
        StructField("target_config", MapType(StringType(), StringType()), True),
        StructField("fallback_config", MapType(StringType(), StringType()), True),
        StructField("last_modified", TimestampType(), True)
    ])

    if not table_exists:
        print(f"Tabela {db_name}.{table_name_control} não existe. Criando...")

        # DataFrame vazio para inicializar Delta log
        df_empty = spark.createDataFrame([], delta_schema)
        df_empty.write.format("delta").mode("overwrite").save(delta_table_path)

        # Cria tabela no metastore (sem comentários)
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {db_name}.{table_name_control}
        USING DELTA
        LOCATION '{delta_table_path}'
        """
        spark.sql(create_sql)
        print(f"Tabela {db_name}.{table_name_control} criada com sucesso.")

    else:
        print(f"Tabela {db_name}.{table_name_control} já existe. Apenas inserindo dados...")

    # Inserção dos dados
    data = [(version, source_layer, table_name_target, schema_array, schema_target, schema_depara, regras_array,
             source_config, target_config, fallback_config, datetime.now())]
    df_insert = spark.createDataFrame(data, delta_schema)
    df_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{db_name}.{table_name_control}")

    print(f"Contrato inserido com sucesso na tabela {db_name}.{table_name_control}.")


schema_array = [
    {
        "name_column": "author_ura",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "author_name",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "author_label",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "updated_at",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "rating",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "version",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "review_id",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "title",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "content",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "content_type_attribute",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "link_rel",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "link_href",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "vote_sum",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "content_term",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "content_label",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "vote_count",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "app_reference",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "ingestion_ts",
        "type_column": "TIMESTAMP",
        "other": "apple_reviews"
    },
    {
        "name_column": "date_load",
        "type_column": "STRING",
        "other": "apple_reviews"
    },
    {
        "name_column": "submission_date",
        "type_column": "TIMESTAMP",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "client_id",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "segment",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "classification",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "feedback_rating",
        "type_column": "INT",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "feedback_comment",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "service_type",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "service_id",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "feedback_specific_to_service",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "source_channel",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "app_reference",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "source_id",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "user_agent",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "ingestion_ts",
        "type_column": "TIMESTAMP",
        "other": "internaldb_reviews"
    },
    {
        "name_column": "date_load",
        "type_column": "STRING",
        "other": "internaldb_reviews"
    }
]

schema_target = [
  {
    "name_column": "review_id",
    "type_column": "STRING",
    "comment": "Identificador único da avaliação, seja original ou gerado para unificação."
  },
  {
    "name_column": "client_id",
    "type_column": "STRING",
    "comment": "Identificador do cliente ou autor da avaliação."
  },
  {
    "name_column": "review_date",
    "type_column": "TIMESTAMP",
    "comment": "Data da publicação ou envio da avaliação."
  },
  {
    "name_column": "review_rating",
    "type_column": "INT",
    "comment": "Nota da avaliação (ex: 1 a 5), padronizada para inteiro."
  },
  {
    "name_column": "review_title",
    "type_column": "STRING",
    "comment": "Título da avaliação, se aplicável."
  },
  {
    "name_column": "review_text",
    "type_column": "STRING",
    "comment": "Conteúdo principal ou comentário da avaliação."
  },
  {
    "name_column": "review_version",
    "type_column": "STRING",
    "comment": "Versão do aplicativo ou serviço avaliado."
  },
  {
    "name_column": "source_channel",
    "type_column": "STRING",
    "comment": "Canal de origem da avaliação (ex: 'App Store', 'Website')."
  },
  {
    "name_column": "source_system",
    "type_column": "STRING",
    "comment": "Sistema de origem da avaliação (ex: 'apple_reviews', 'internaldb_reviews')."
  },
  {
    "name_column": "segment",
    "type_column": "STRING",
    "comment": "Segmento de cliente, se disponível."
  },
  {
    "name_column": "service_type",
    "type_column": "STRING",
    "comment": "Tipo de serviço avaliado, se aplicável."
  },
  {
    "name_column": "ingestion_ts",
    "type_column": "TIMESTAMP",
    "comment": "Timestamp da ingestão do dado na camada Bronze."
  },
  {
    "name_column": "date_load",
    "type_column": "STRING",
    "comment": "Data da carga, utilizada para particionamento."
  },
  {
    "name_column": "user_agent",
    "type_column": "STRING",
    "comment": "Informações de identificação do navegador/dispositivo."
  }
]

schema_depara = [
]

regras_array = [
  {
    "rule": "not_null",
    "value": "true"
  },
  {
    "rule": "min_value",
    "column": "review_rating",
    "value": "1"
  },
  {
    "rule": "max_value",
    "column": "review_rating",
    "value": "5"
  },
  {
    "rule": "days_back",
    "column": "date_load",
    "value": 365
  }
]

source_config = {
    "format": "delta"
}

target_config = {
    "path_blob": "abfss://s-compass@compassdataprod.dfs.core.windows.net/",
    "directory": "s-compass",
    "format": "delta",
    "mode": "overwrite",
    "partitionBy": "date_load"
}


fallback_config = {
    "create_empty_if_missing": "true"
}

# Parametros indicador para a tabela de controle
container_system = "system-compass"
db_name = "metadata_compass"
table_name = "data_params"

delta_table_path = f"abfss://{container_system}@compassdataprod.dfs.core.windows.net/{container_system}/{table_name}/"

create_or_append_contract(
    delta_table_path=delta_table_path,
    db_name=db_name,
    table_name_control=table_name,
    version=1.1,
    source_layer="s_compass",
    table_name_target="instituicao_reviews",
    schema_array=schema_array,
    schema_target=schema_target,
    schema_depara=schema_depara,
    regras_array=regras_array,
    source_config=source_config,
    target_config=target_config,
    fallback_config=fallback_config
)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType, DoubleType, MapType
from datetime import datetime

def create_or_append_contract(delta_table_path: str,
                              db_name: str,
                              table_name_control: str,
                              version: float,
                              source_layer: str,
                              table_name_target: str,
                              schema_array: list,
                              schema_target: list,
                              schema_depara: list,
                              regras_array: list,
                              source_config: dict,
                              target_config: dict,
                              fallback_config: dict):
    """
    Cria a tabela Delta caso não exista e insere um contrato de dados,
    incluindo configs de fonte, destino e fallback.
    Se a tabela já existir, apenas insere os dados.
    """

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

    # Verifica se a tabela existe
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {db_name}.{table_name_control}")
        table_exists = True
    except Exception:
        table_exists = False

    # Schema Delta nativo
    delta_schema = StructType([
        StructField("version", DoubleType(), True),
        StructField("source_layer", StringType(), True),
        StructField("table_name_target", StringType(), True),
        StructField("schema_expected", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True),
                StructField("other", StringType(), True) 
            ])
        ), True),
        StructField("schema_target", ArrayType(
            StructType([
                StructField("name_column", StringType(), True),
                StructField("type_column", StringType(), True)
            ])
        ), True),
        StructField("schema_depara", ArrayType(
            StructType([
                StructField("source_column", StringType(), True),
                StructField("target_column", StringType(), True)
            ])
        ), True),
        StructField("rule_control", ArrayType(
            StructType([
                StructField("rule", StringType(), True),
                StructField("value", StringType(), True)
            ])
        ), True),
        StructField("source_config", MapType(StringType(), StringType()), True),
        StructField("target_config", MapType(StringType(), StringType()), True),
        StructField("fallback_config", MapType(StringType(), StringType()), True),
        StructField("last_modified", TimestampType(), True)
    ])

    if not table_exists:
        print(f"Tabela {db_name}.{table_name_control} não existe. Criando...")

        # DataFrame vazio para inicializar Delta log
        df_empty = spark.createDataFrame([], delta_schema)
        df_empty.write.format("delta").mode("overwrite").save(delta_table_path)

        # Cria tabela no metastore (sem comentários)
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {db_name}.{table_name_control}
        USING DELTA
        LOCATION '{delta_table_path}'
        """
        spark.sql(create_sql)
        print(f"Tabela {db_name}.{table_name_control} criada com sucesso.")

    else:
        print(f"Tabela {db_name}.{table_name_control} já existe. Apenas inserindo dados...")

    # Inserção dos dados
    data = [(version, source_layer, table_name_target, schema_array, schema_target, schema_depara, regras_array,
             source_config, target_config, fallback_config, datetime.now())]
    df_insert = spark.createDataFrame(data, delta_schema)
    df_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{db_name}.{table_name_control}")

    print(f"Contrato inserido com sucesso na tabela {db_name}.{table_name_control}.")



schema_array = []

schema_target = [
    {"name_column": "review_year", "type_column": "STRING", "comment": "Ano da avaliação"},
    {"name_column": "review_month", "type_column": "STRING", "comment": "Mês da avaliação"},
    {"name_column": "review_version", "type_column": "STRING", "comment": "Versão do aplicativo na avaliação"},
    {"name_column": "segment", "type_column": "STRING", "comment": "Segmento do cliente"},
    {"name_column": "service_type", "type_column": "STRING", "comment": "Tipo de serviço avaliado"},
    {"name_column": "app_reference", "type_column": "STRING", "comment": "Aplicativo de origem da avaliação"},
    {"name_column": "review_count", "type_column": "LONG", "comment": "Contagem de avaliações no grupo"},
    {"name_column": "average_rating", "type_column": "DOUBLE", "comment": "Nota média das avaliações do grupo"},
    {"name_column": "min_rating", "type_column": "DOUBLE", "comment": "Nota mínima do grupo"},
    {"name_column": "max_rating", "type_column": "DOUBLE", "comment": "Nota máxima do grupo"},
    {"name_column": "nps_score", "type_column": "DOUBLE", "comment": "NPS (Net Promoter Score) do grupo"}
]


schema_depara = []

regras_array = [
    
        {"rule": "not_empty", "value": "true"},
        {"rule": "evolution_mergeschema", "value": "false"}
   
]

source_config = {
    "format": "delta"
}

target_config = {
    "path_blob": "abfss://g-compass@compassdataprod.dfs.core.windows.net/",
    "directory": "g-compass",
    "format": "delta",
    "mode": "overwrite"
}

fallback_config = {
    "create_empty_if_missing": "true"
}

# Parametros indicador para a tabela de controle
container_system = "system-compass"
db_name = "metadata_compass"
table_name = "data_params"

delta_table_path = f"abfss://{container_system}@compassdataprod.dfs.core.windows.net/{container_system}/{table_name}/"

create_or_append_contract(
    delta_table_path=delta_table_path,
    db_name=db_name,
    table_name_control=table_name,
    version=1.1,
    source_layer="g_compass",
    table_name_target="reviews_customer_compass",
    schema_array=schema_array,
    schema_target=schema_target,
    schema_depara=schema_depara,
    regras_array=regras_array,
    source_config=source_config,
    target_config=target_config,
    fallback_config=fallback_config
)


In [0]:
%sql
SELECT * FROM metadata_compass.data_params